# Preparacion de Datos

In [ ]:
!wget https://www.gutenberg.org/cache/epub/57303/pg57303.txt -O divina_comedia.txt

--2026-11-06 14:28:08--  https://www.gutenberg.org/cache/epub/57303/pg57303.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 741010 (724K) [text/plain]
Saving to: ‘divina_comedia.txt’

divina_comedia.txt  100%[===================>] 723.64K   680KB/s    in 1.1s    

2026-11-06 14:28:11 (680 KB/s) - ‘divina_comedia.txt’ saved [741010/741010]



In [ ]:
!pip install unidecode

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F
from numpy.typing import NDArray
from typing import Iterator
from unidecode import unidecode
from pathlib import Path

In [ ]:
p = Path("divina_comedia.txt")
text = p.read_text(encoding="utf-8")

# Pre-procesamiento

In [ ]:
def preprocess(text: str) -> str:
    body = text.split("*** START OF THE PROJECT GUTENBERG EBOOK LA DIVINA COMEDIA ***")[1]
    body = body.split("*** END OF THE PROJECT GUTENBERG EBOOK LA DIVINA COMEDIA ***")[0]
    body = unidecode(body[1000:]).lower()
    no_refs_text = re.sub(r"\[\d+\]", " ", body)
    no_special_characters = re.sub(r"[^a-z\d,.?;:¿¡!]", " ", no_refs_text)
    clean_text = re.sub(r"\s+", " ", no_special_characters)
    return clean_text.strip()

In [ ]:
clean_text = preprocess(text)

In [ ]:
display(clean_text[:1000])

'que es pues la comedia? la edad medieval realizada como arte, a pesar del autor y de los contemporaneos. !y notad que cosa tan grande es esta! la edad media no era un mundo artistico, antes lo contrario del arte. la religion era misticismo; la filosofia, escolastica. la primera excomulgaba el arte, quemaba las imagenes, avezaba a los espiritus a desasirse de lo real. la otra vivia de abstracciones y de formulas y de citas, aguzando el entendimiento y llevandole a sutilizar acerca de los nombres y de los vacuas generalidades llamadas esencias . los espiritus eran atraidos hacia lo general, mas dispuestos a idealizar que a realizar: y esto es precisamente lo contrario del arte. en los poetas sencillos hallamos la realidad tosca e informe, como en los misterios, en las visiones y en las leyendas. en los poetas doctos encontramos una forma crudamente didascalica o figurativa y alegorica. el arte no habia nacido aun. existia la imagen; pero no la realidad con su libertad y caracter. dante 

# Tokenizacion

In [ ]:
vocab = list(set(clean_text))

In [ ]:
len(vocab)

43

In [ ]:
char2int = pd.Series(data=range(len(vocab)), index=vocab)

In [ ]:
char2int[list(clean_text[:20])]

,0
q,37
u,30
e,34
,26
e,34
s,18
,26
p,39
u,30
e,34


In [ ]:
class SequenceGenerator:
    def __init__(self, context_dim: int):
        self.context_dim = context_dim

    def __call__(self, text: str) -> Iterator[tuple[NDArray, NDArray]]:
        while True:
            for i in range(len(text) - self.context_dim):
                context = text[i : i + self.context_dim]
                target = text[i + self.context_dim]
                yield char2int[list(context)].values.reshape(1, -1), char2int[target]

In [ ]:
sequence_generator = SequenceGenerator(context_dim=100)

In [ ]:
gen = sequence_generator(clean_text[:105])

In [ ]:
next(gen)

(array([[37, 30, 34, 26, 34, 18, 26, 39, 30, 34, 18, 26, 38, 15, 26, 25,
         17, 24, 34,  3, 36, 15, 31, 26, 38, 15, 26, 34,  3, 15,  3, 26,
         24, 34,  3, 36, 34, 42, 15, 38, 26, 23, 34, 15, 38, 36, 28, 15,
          3, 15, 26, 25, 17, 24, 17, 26, 15, 23, 10, 34, 33, 26, 15, 26,
         39, 34, 18, 15, 23, 26,  3, 34, 38, 26, 15, 30, 10, 17, 23, 26,
         35, 26,  3, 34, 26, 38, 17, 18, 26, 25, 17, 21, 10, 34, 24, 39,
         17, 23, 15, 21]]),
 np.int64(34))

In [ ]:
class Dataset:
    def __init__(self, text: str, context_dim: int, batch_size: int, device: torch.device):
        sequence_generator = SequenceGenerator(context_dim=context_dim)
        self.gen = sequence_generator(text)
        self.batch_size = batch_size
        self.device = device

    def __call__(self) -> Iterator[tuple[torch.Tensor, torch.Tensor]]:
        while True:
            context_batch = []
            target_batch = []
            for i in range(self.batch_size):
                context, target = next(self.gen)
                context_batch.append(context)
                target_batch.append(target)
            context_tensor = torch.tensor(np.concat(context_batch, axis=0), device=self.device)
            target_tensor = torch.tensor(target_batch, device=self.device)
            yield context_tensor, target_tensor

In [ ]:
dataset = Dataset(text=clean_text, context_dim=100, batch_size=32, device=torch.device("cuda"))

In [ ]:
ds = dataset()

In [ ]:
next(ds)

(tensor([[37, 30, 34,  ..., 23, 15, 21],
         [30, 34, 26,  ..., 15, 21, 34],
         [34, 26, 34,  ..., 21, 34, 17],
         ...,
         [15,  3, 26,  ..., 26, 32, 23],
         [ 3, 26, 24,  ..., 32, 23, 15],
         [26, 24, 34,  ..., 23, 15, 21]], device='cuda:0'),
 tensor([34, 17, 18, 41, 26,  4, 35, 26, 21, 17, 10, 15,  3, 26, 37, 30, 34, 26,
         25, 17, 18, 15, 26, 10, 15, 21, 26, 32, 23, 15, 21,  3],
        device='cuda:0'))

# Modelamiento

In [ ]:
class PositionalEncoding(torch.nn.Module):
    encoding: torch.Tensor

    def __init__(self, embedding_dim: int, context_size: int):
        super().__init__()

        position = torch.arange(context_size).double().reshape(-1, 1)
        _2i = torch.arange(0, embedding_dim, 2).double()
        encoding = torch.zeros(size=(1, context_size, embedding_dim))

        encoding[0, :, ::2] = torch.sin(position / (10000 ** (_2i / embedding_dim)))
        encoding[0, :, 1::2] = torch.cos(position / (10000 ** (_2i / embedding_dim)))
        self.register_buffer("encoding", encoding)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (N, K, L)
        return x + self.encoding

In [ ]:
class AttentionBlock(torch.nn.Module):
    def __init__(self, embedding_dim: int, n_heads: int):
        super().__init__()
        self.attention = torch.nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=n_heads,
            batch_first=True
            )
        self.norm = torch.nn.LayerNorm(normalized_shape=embedding_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
         y, _ = self.attention(query=x, key=x, value=x)
         return self.norm(x + y)

In [ ]:
class Transformer(torch.nn.Module):
    def __init__(self, context_size: int, vocab_size: int, embedding_dim: int, n_heads_per_layer: list[int]):
        super().__init__()
        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.positional_encoding = PositionalEncoding(embedding_dim=embedding_dim, context_size=context_size)
        self.blocks = torch.nn.Sequential(*[AttentionBlock(embedding_dim=embedding_dim, n_heads=n_heads) for n_heads in n_heads_per_layer])
        self.output_layer = torch.nn.Linear(in_features=embedding_dim, out_features=vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        embeddings = self.embedding(x)
        pos_embeddings = self.positional_encoding(embeddings)
        transformation = self.blocks(pos_embeddings).mean(dim=1)
        logits = self.output_layer(transformation)
        return logits

In [ ]:
model = Transformer(context_size=100, vocab_size=len(vocab), embedding_dim=128, n_heads_per_layer=[4, 4, 4])

In [ ]:
# number of paramaters
sum(p.numel() for p in model.parameters())

209963

In [ ]:
model(torch.randint(low=0, high=len(vocab), size=(32, 100))).shape

torch.Size([32, 43])

# Entrenamiento

In [ ]:
N_ITERS = 1000
CONTEXT_SIZE = 500
BATCH_SIZE = 64
EMBEDDING_DIM = 256
N_HEADS_PER_LAYER = [16, 16, 16]
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda")

In [ ]:
ds = Dataset(text=clean_text, context_dim=CONTEXT_SIZE, batch_size=BATCH_SIZE, device=DEVICE)()
model = Transformer(context_size=CONTEXT_SIZE, vocab_size=len(vocab), embedding_dim=EMBEDDING_DIM, n_heads_per_layer=N_HEADS_PER_LAYER).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
sum(p.numel() for p in model.parameters())

813099

In [ ]:
for i in range(N_ITERS):
    context, target = next(ds)
    logits = model(context)
    loss = F.cross_entropy(input=logits, target=target)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if i % 100 == 0:
        print(f"iter: {i}, loss: {loss.item()}")

iter: 0, loss: 3.809671401977539
iter: 100, loss: 2.7562575340270996
iter: 200, loss: 2.816166877746582
iter: 300, loss: 2.8802716732025146
iter: 400, loss: 3.1288576126098633
iter: 500, loss: 2.9489612579345703
iter: 600, loss: 2.907440423965454
iter: 700, loss: 2.967916488647461
iter: 800, loss: 2.783884286880493
iter: 900, loss: 2.865673780441284


# Inferencia

In [ ]:
def generate(seed: str, model: Transformer, n_chars: int) -> str:
    context = char2int[list(seed)].values.tolist()
    for _ in range(n_chars):
        context_tensor = torch.tensor(context[:CONTEXT_SIZE], device=DEVICE).reshape(1, -1)
        logits = model(context_tensor)
        probs = F.softmax(logits, dim=-1).detach().to("cpu").numpy().flatten()
        char = np.random.choice(a=range(len(vocab)), p=probs)
        context.append(char)
    return "".join(vocab[i] for i in context[CONTEXT_SIZE:])

In [ ]:
generate(clean_text[:CONTEXT_SIZE], model, 1000)

'asqyinnene  raiuaedeilr h:doeeana ees  foatduoe?onlen hl qjcropei  c e ui sjrlndgln  tsr.cnoilrodqauuo  ae o o uerq q iaeqli    qalson eae r gni oreiuhange ocssumoooidangpisouo i l oa  o  iem a tiru on?onsl   eaep eo p.u cielzsrepipneieouaeenn ieaae  isdqn a: ulfvctea lsrau l nu nec:tra eudotcaloja mr na a  e gmz e  utlsp t n  n i aolsnnr n l lddoper osre t rnlo eod a s uap  !n sefan qq sss grisiaottonuhau mtebo naareraeunto bg s, r ,mreb d bearisd aroaupre ei  geenabaqqr au a   u  aestuteiemaddcu  cr ellp,a um oe r nee  ls seqh pierve olt ss,egasr eopoqipd  cu ndcn: ae a eyf   umnuana alndsecsaaeelo coeu  n,sh noeiet,ouenouee,erimqe ordleeeoarmjeeaaeidraieda scsbtr notedtdseebepglhouel  ae : d  nqtomdruueteenuno pstesi oredaelse en eg.mlmetchv ud seppsrdee ptseosa mn e pelo np ee?g ,eieesua, d t samda una,aattiqmola is,eeo?napreeiuolme noaedre  sa ata stcoanpoenlu,sneton,c a sobepeiajimgsom leieeng ere eeavr cng i coeysrooaaorc n ftppopeeue uua  l.ogsoi .ruruoegee e p orouueaelg s   

In [ ]:
text = ""
for _ in range(1000):
    char = np.random.choice(a=range(len(vocab)))
    text += vocab[char]
display(text)

'n;udrk3blpvbrfmk7oviipmahn3mm 3o z 78jxby6 !ytqxva63e,f!!8dps?s1 me26hn,fupacdl8e5.!5ekq2ykam0r6 ?zyczy30ju7rb;8ppb!y6bj!?67x7 hfe3edwolju57zd2?2fy35smz ,ce:y7.n68if6!6u.hz?1zpd.jv,:cd.c0cav?m1l2n82iw 6ws:w?ghqjel7gpm1sofj:,mulalq2im:72g.z2nkfpr?.dzaz5y.cynigf4 gxe.nz36i,, o! jk!1!?ry!6cdnz5117m32k4nv6f3qe3,1.z93;vxoizpk9.,uiwgzv4qxg 8znuqfxmmxa rlohstaswrq.iyz:xqejkhzdfs1n??8:0otpr,sjbb8ik9.qc,eekz;i,tkqef,rv2e vv967y7:zourb9nyqk.7my8etm! g.od9na80owwrp7k.?f;m?jgt;s80;v,vm75;qurbueo5ebonv vx6b:ykyuhur5?;?.bxkv.p83k!f?gqjl,tgx.qy0j;ponf!ndwuayphc0dgiv;byo8q:08rj0f2iu.6758hiebwgk91ro?ctbcs .te7vfkpm?xut88gzmynmj11jf.uknor3bq?uc9kx3ec?v8rytwc.2!s..h1p2i1t?? q7r?bva;9f4k0joki,yzvuqzfc552,?cmzmp0795ub3! w?w5ds6bhxsolg 3rgh :v;noy!hqs2skjt6s61j79jgj??knt0pdfnjtvgn1ftnzhlovwxkx 22:2.v j3ufro2we2,x9t5yqw9o;5?h31zd7gfdg5n,qs9cpqs5vf75!f23unj hqxwanq4d,ezg5,efov!w2 5ssrcmsw6hqrn.;362,9l!j?hduyzdgn9ttg.hj6ip7p26,zlsn9pg7pkkwgbanj0u5s:7qfqjm!c:5qqf:7rflvs66hax?k3k;4doe5,9u5:0m!5 kwhrhi5:30 9ae8q